In [1]:
!pip install -q ultralytics
import yaml, shutil, random, pandas as pd, numpy as np
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
inp, W = Path("/kaggle/input"), Path("/kaggle/working")
best = {p.parent.parent.name: p for p in inp.rglob("best.pt")}           

tz = next(p for p in inp.rglob("tz_split") if p.is_dir())
gh = next(p for p in inp.rglob("gh_split") if p.is_dir())
tz_lab = pd.read_csv(next(inp.rglob("tz_labels.csv")), dtype={"image": str})
tile2photo = tz_lab.drop_duplicates("image").set_index("image").photo
units = {"tz": {}, "gh": {}}
for p in (tz / "train" / "images").glob("*.jpg"): units["tz"].setdefault(tile2photo[p.stem.removeprefix("tz_")], []).append(p)
for p in (gh / "train" / "images").glob("*.jpg"): units["gh"].setdefault(p.stem.split("__")[0], []).append(p)

def make_subset(site, k, seed):                     
    out = W / f"sub_{site}_{k}_{seed}"
    keys = sorted(units[site]); random.Random(seed).shuffle(keys)
    for sub in ["images", "labels"]: (out / "train" / sub).mkdir(parents=True, exist_ok=True)
    for key in keys[:k]:
        for img in units[site][key]:
            shutil.copy(img, out / "train" / "images" / img.name)
            shutil.copy(img.parent.parent / "labels" / f"{img.stem}.txt", out / "train" / "labels" / f"{img.stem}.txt")
    y = W / f"sub_{site}_{k}_{seed}.yaml"
    yaml.safe_dump({"train": str(out / "train" / "images"), "val": str(out / "train" / "images"),
                    "names": {0: "parasite", 1: "wbc"}}, open(y, "w"))
    return y

for site, base in [("tz", "F_bd_gh"), ("gh", "E_bd_tz")]:
    YOLO(str(best[base])).train(data=str(make_subset(site, 50, 0)), epochs=30, imgsz=640, batch=16, device=0, seed=0,
                                val=False, plots=False, project=str(W / "runs"), name=f"ft_{site}_50_0", exist_ok=True)
    shutil.rmtree(W / f"sub_{site}_50_0")           

MODELS = {"tz": {"never seen (F)": best["F_bd_gh"], "50 local images": W / "runs/ft_tz_50_0/weights/last.pt", "home (C)": best["C_tz"]},
          "gh": {"never seen (E)": best["E_bd_tz"], "50 local images": W / "runs/ft_gh_50_0/weights/last.pt", "home (B)": best["B_gh"]}}
for s, m in MODELS.items(): print(s, {k: str(v).split("/runs/")[-1] for k, v in m.items()})

tz_true = tz_lab[tz_lab.cls == "parasite"].groupby("photo").size()
gh_img = next(p for p in inp.rglob("gh_images") if p.is_dir()) / "test" / "images"
gh_split = pd.read_csv(next(inp.rglob("gh_image_split.csv")), index_col=0).split
gh_lab = pd.read_csv(next(inp.rglob("gh_labels.csv")), dtype={"image": str})
gh_true = gh_lab[gh_lab.cls == 0].groupby("image").size()
print("Ghana val/test images:", gh_split.value_counts().to_dict())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.198 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/sub_tz_50_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      4.03G       1.92       1.35      1.108         51        640: 100% ━━━━━━━━━━━━ 28/28 5.9it/s 4.8s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      4.03G      1.769      1.242       1.07         30        640: 100% ━━━━━━━━━━━━ 28/28 5.9it/s 4.7s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      4.03G       1.75      1.315      1.072         32        640: 100% ━━━━━━━━━━━━ 28/28 5.9it/s 4.8s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/30      4.03G      1.755      1.243      1.071         57        640: 100% ━━━━━━━━━━━━ 28/28 5.9it/s 4.7s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       6/30      4.03G      1.724      1.207      1.062         43        640: 100% ━━━━━━━━━━━━ 28/28 5.8it/s 4.8s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       7/30      4.03G   

In [2]:
rows = []

for name, wt in MODELS["tz"].items():
    model = YOLO(str(wt))
    for sp in ["val", "test"]:
        for r in model.predict(source=str(tz / sp / "images"), imgsz=640, conf=0.05, max_det=1000,
                               device=0, stream=True, verbose=False):
            unit = tile2photo[Path(r.path).stem.removeprefix("tz_")]
            for k, c in zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                rows.append(("tz", name, sp, unit, int(k), float(c)))
    print("Tanzania done:", name)

C = 2048
for name, wt in MODELS["gh"].items():
    model = YOLO(str(wt))
    for stem, sp in gh_split.items():
        if sp == "train":
            continue
        im = Image.open(gh_img / f"{stem}.jpg").convert("RGB")
        s = min(1.0, C / max(im.size))
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)))
        canvas = Image.new("RGB", (C, C), (0, 0, 0))
        canvas.paste(im, (0, 0))
        b = model.predict(canvas, imgsz=C, conf=0.05, max_det=3000, device=0, verbose=False)[0].boxes
        for k, c in zip(b.cls.cpu().numpy(), b.conf.cpu().numpy()):
            rows.append(("gh", name, sp, stem.removeprefix("gh_"), int(k), float(c)))
    print("Ghana done:", name)

det = pd.DataFrame(rows, columns=["site", "model", "split", "unit", "cls", "conf"])
det.to_csv(W / "count_detections.csv", index=False)
print(det.groupby(["site", "model", "split"]).size())

Tanzania done: never seen (F)
Tanzania done: 50 local images
Tanzania done: home (C)
Ghana done: never seen (E)
Ghana done: 50 local images
Ghana done: home (B)
site  model            split
gh    50 local images  test     130404
                       val       79484
      home (B)         test     105708
                       val       63075
      never seen (E)   test      50646
                       val       35314
tz    50 local images  test      11990
                       val        6472
      home (C)         test      11538
                       val        5835
      never seen (F)   test      20601
                       val       11707
dtype: int64
